In [1]:
import numpy as np
import netket as nk
from math import comb
from scipy.sparse.linalg import eigsh
from netket.experimental.operator import ParticleNumberAndSpinConservingFermioperator2nd

# 为了代码简洁，直接重命名算符，与你的成功案例保持一致
from netket.operator.fermion import create as cdag
from netket.operator.fermion import destroy as c
from netket.operator.fermion import number as nc

# --- 1. 系统参数配置 ---
L = 5                 # 一维链长度
t = 1.0               # 跳跃能级
U = 4.0               # 原位排斥能 (Hubbard U)
n_orbitals = L


N_up = 2
N_dn = 2

# --- 2. 构造【受限的】费米子希尔伯特空间 ---
# 【核心修复】：使用 n_fermions_per_spin 来分别固定上下自旋的电子数
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals, s=1/2, n_fermions_per_spin=(N_up, N_dn)
)

print('n_orbitals =', n_orbitals)
print('fixed particles (N_up, N_dn) =', (N_up, N_dn))

# --- 3. 构造哈密顿量 ---
H = 0.0  

# 动能项 (Hopping) - OBC 开边界
# sum_{i, sigma} -t * (c^dag_{i, sigma} c_{i+1, sigma} + h.c.)
for i in range(L - 1):
    for sz in (+1, -1):
        H += -t * (cdag(hi, i, sz=sz) @ c(hi, i+1, sz=sz))
        H += -t * (cdag(hi, i+1, sz=sz) @ c(hi, i, sz=sz))

# 相互作用项 (Interaction)
# sum_{i} U * n_{i, up} * n_{i, down}
for i in range(L):
    n_up_i = nc(hi, i, sz=+1)
    n_dn_i = nc(hi, i, sz=-1)
    H += U * (n_up_i @ n_dn_i)

# --- 4. 维度评估与守恒量扇区构建 ---
sector_dim = comb(n_orbitals, N_up) * comb(n_orbitals, N_dn)
max_dim_for_sparse_ed = 2000000  

print("\n--- 系统信息 ---")
print(f"L={L}, U={U}, 电子配置=(N_up={N_up}, N_dn={N_dn})")
print(f"预计该扇区希尔伯特空间维度 (Sector Dim) = {sector_dim}")

if sector_dim > max_dim_for_sparse_ed:
    print("[Skip ED] 该扇区维度过大，不执行 to_sparse()/eigsh 以避免内存溢出。")
else:
    # --- 5. 提取守恒子空间并严格对角化 ---
    # 直接传入 H，底层会自动读取 hi 中的 n_fermions_per_spin 约束
    H_ed = ParticleNumberAndSpinConservingFermioperator2nd.from_fermionoperator2nd(H)
    
    print("\n正在构建稀疏矩阵...")
    sp_h = H_ed.to_sparse()
    
    print("正在使用 Lanczos 算法求解基态...")
    # 求解前 k 个最小特征值 (SA: Smallest Algebraic)
    k_vals = min(4, sp_h.shape[0] - 2)
    eig_vals, eig_vecs = eigsh(sp_h, k=k_vals, which="SA")
    
    # 特征值从小到大排序
    eig_vals = np.sort(np.real(eig_vals))

# --- 5. 计算并输出结果 ---
    print("\n--- 计算结果 ---")
    print(f"当前守恒扇区的矩阵实际维度: {sp_h.shape[0]}")
    
    # 基态能量 E0
    e0 = eig_vals[0]
    # 单位格点平均能量
    e_per_site = e0 / L

    print(f"单位格点平均能量 E0/L = {e_per_site:.16f}")
    print(f"最低 {len(eig_vals)} 个能级 = {eig_vals}")

C:\Users\10783\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: With many Markov Chains (e.g GPUs), n_discard_per_chain>5 is often inefficient.

n_orbitals = 5
fixed particles (N_up, N_dn) = (2, 2)

--- 系统信息 ---
L=5, U=4.0, 电子配置=(N_up=2, N_dn=2)
预计该扇区希尔伯特空间维度 (Sector Dim) = 100

正在构建稀疏矩阵...
正在使用 Lanczos 算法求解基态...

--- 计算结果 ---
当前守恒扇区的矩阵实际维度: 100
单位格点平均能量 E0/L = -0.6846901197768027
最低 4 个能级 = [-3.4234506  -2.97161306 -2.49859337 -2.39905888]


In [2]:
import numpy as np

def measure_pairing_correlation(hi, psi, i, j, k, l):
    """
    测量配对关联函数 P(ij, kl) = ⟨Ψ| Δ†_ij Δ_kl |Ψ⟩
    其中 Δ_xy = c_{x↑} c_{y↓} - c_{x↓} c_{y↑}
    
    展开后共计 4 项：
    P(ij, kl) = -(c†_{i↑} c†_{j↓} - c†_{i↓} c†_{j↑}) @ (c_{k↑} c_{l↓} - c_{k↓} c_{l↑})
              = - c†_{i↑} c†_{j↓} c_{k↑} c_{l↓}  (项1)
                + c†_{i↑} c†_{j↓} c_{k↓} c_{l↑}  (项2)
                + c†_{i↓} c†_{j↑} c_{k↑} c_{l↓}  (项3)
                - c†_{i↓} c†_{j↑} c_{k↓} c_{l↑}  (项4)
    """
    # 物理检查：如果 i==j 或 k==l，由于 Pauli 不相容原理，单重态配对直接为 0
    if i == j or k == l:
        zeros_dict = {
            "T1 (- c†_i↑ c†_j↓ c_k↑ c_l↓)": 0.0,
            "T2 (+ c†_i↑ c†_j↓ c_k↓ c_l↑)": 0.0,
            "T3 (+ c†_i↓ c†_j↑ c_k↑ c_l↓)": 0.0,
            "T4 (- c†_i↓ c†_j↑ c_k↓ c_l↑)": 0.0,
            "Total": 0.0
        }
        return 0.0, zeros_dict
        
    # 定义基础单体算符
    cdag_i_up = cdag(hi, i, sz=+1)
    cdag_j_dn = cdag(hi, j, sz=-1)
    cdag_i_dn = cdag(hi, i, sz=-1)
    cdag_j_up = cdag(hi, j, sz=+1)
    
    c_k_up = c(hi, k, sz=+1)
    c_l_dn = c(hi, l, sz=-1)
    c_k_dn = c(hi, k, sz=-1)
    c_l_up = c(hi, l, sz=+1)
    
    # 分别构造 4 个分量算符 (直接将理论符号乘在算符前面)
    term1_op = -1.0 * (cdag_i_up @ cdag_j_dn @ c_k_up @ c_l_dn)
    term2_op =  1.0 * (cdag_i_up @ cdag_j_dn @ c_k_dn @ c_l_up)
    term3_op =  1.0 * (cdag_i_dn @ cdag_j_up @ c_k_up @ c_l_dn)
    term4_op = -1.0 * (cdag_i_dn @ cdag_j_up @ c_k_dn @ c_l_up)
    
    # 辅助函数：求单个算符在守恒空间中的期望值
    def get_expectation(op):
        op_ed = ParticleNumberAndSpinConservingFermioperator2nd.from_fermionoperator2nd(op)
        sp_op = op_ed.to_sparse()
        return np.real(np.vdot(psi, sp_op.dot(psi)))
    
    # 计算各项的值
    val1 = get_expectation(term1_op)
    val2 = get_expectation(term2_op)
    val3 = get_expectation(term3_op)
    val4 = get_expectation(term4_op)
    
    # 总值
    total_val = val1 + val2 + val3 + val4
    
    # 将结果装入带标签的字典
    details = {
        "T1 (- c†_i↑ c†_j↓ c_k↑ c_l↓)": val1,
        "T2 (+ c†_i↑ c†_j↓ c_k↓ c_l↑)": val2,
        "T3 (+ c†_i↓ c†_j↑ c_k↑ c_l↓)": val3,
        "T4 (- c†_i↓ c†_j↑ c_k↓ c_l↑)": val4,
        "Total": total_val
    }
    
    return total_val, details

In [7]:
# 提取基态波函数
psi0 = eig_vecs[:, 0]

# 调用我们刚才修改好的函数，提取出总值和 4 个分项
val, details = measure_pairing_correlation(hi, psi0, 0, 1, 2 , 3)

# 打印出极度舒适的排版结果
print("\n==== Python ED: Pairing 测量详情 P(0,1; 2,4) ====")
for formula, v in details.items():
    print(f"{formula:35s} = {v:15.8f}")
print("==================================================\n")


==== Python ED: Pairing 测量详情 P(0,1; 2,4) ====
T1 (- c†_i↑ c†_j↓ c_k↑ c_l↓)        =     -0.00312619
T2 (+ c†_i↑ c†_j↓ c_k↓ c_l↑)        =     -0.02273734
T3 (+ c†_i↓ c†_j↑ c_k↑ c_l↓)        =     -0.02273734
T4 (- c†_i↓ c†_j↑ c_k↓ c_l↑)        =     -0.00312619
Total                               =     -0.05172705

